In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# 🔖 Project Metadata (Capstone Info)
project_title = "Travel Agent AI with Google GenAI"
submitted by = "Pervaz Md Shoriful Hussain"
submission_date = "April 19, 2025"
tools_used = ["Gemini 2.0 Flash", "ChromaDB", "Google Search", "Python", "Kaggle"]



# ✨ Google Gen AI Capstone Project: Travel Agent AI

# 🔍 Introduction
```markdown
In this project, we build a Travel Agent AI using Gemini 2.0 Flash. The AI can:
- Plan travel itineraries using few-shot and chain-of-thought prompting
- Answer travel-related questions from a custom guide (RAG)
- Use Google Search grounding to enhance with real-time insights

Technologies used: Gemini 2.0, ChromaDB, Python, Google Search API
```

# 📦 Installation and Setup
```markdown
We begin by installing necessary packages and authenticating the Gemini API using Kaggle secrets.
```


In [2]:
!pip install -qU google-genai==1.7.0 chromadb==0.6.3

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 3.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.7/144.7 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 611.1/611.1 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 56.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.9/100.9 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.2/284.2 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 58.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.6/101.6 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 79.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.9/55.9 kB 2.6 MB/s eta 0:

In [3]:
import os
import numpy as np
import pandas as pd
from google import genai
from google.genai import types
from google.api_core import retry
import chromadb
from chromadb.api.types import Documents
from chromadb.utils.embedding_functions import EmbeddingFunction
from kaggle_secrets import UserSecretsClient

GOOGLE_API_KEY = UserSecretsClient().get_secret("GOOGLE_API_KEY")
client = genai.Client(api_key=GOOGLE_API_KEY)


# 🧠 Few-Shot Prompting + Chain-of-Thought
```markdown
We use a smart few-shot prompt to guide Gemini in creating structured travel plans, step-by-step.
```

In [4]:
user_trip_prompt = "Plan a 3-day solo cultural trip to Jaipur in February (budget)"

few_shot_prompt = f"""
You are an Indian travel agent AI. Plan trips in day-wise format for solo, couple, or family travelers within India.

Each itinerary should include:
- Morning, Afternoon, Evening plan
- Local food suggestions
- Estimated daily cost (in INR)
- Local travel tips (culture, safety, transport)

---
Example 1:
Trip: 2-day spiritual trip to Varanasi in December (budget)
Response:
Day 1:
- Morning: Sunrise boat ride on the Ganges (~INR 300)
- Lunch: Banarasi thali near Dashashwamedh Ghat (~INR 200)
- Afternoon: Visit Kashi Vishwanath Temple
- Evening: Ganga Aarti at Assi Ghat
- Estimated Total: ~INR 800

---
Now plan this trip:
{user_trip_prompt}
"""

In [5]:
response = client.models.generate_content(
    model="gemini-2.0-flash",
    contents=few_shot_prompt
)
print("📅 Generated Itinerary:\n", response.text)


📅 Generated Itinerary:
 Okay, here's a 3-day solo cultural trip to Jaipur in February, planned with a budget in mind:

**Trip: 3-Day Solo Cultural Trip to Jaipur in February (Budget)**

**Day 1: Exploring the Pink City Core**

*   **Morning (8:00 AM - 12:00 PM):**
    *   Visit Hawa Mahal (Palace of Winds). Arrive early to avoid crowds. (Entry Fee: INR 50 for Indians).
    *   Walk through the Johari Bazaar, known for its jewellery. Practice your bargaining skills!
*   **Afternoon (12:00 PM - 4:00 PM):**
    *   Lunch: Try *Dal Baati Churma* at a local restaurant in Johari Bazaar, like Laxmi Misthan Bhandar (LMB) (Approx. INR 300).
    *   Visit City Palace, a beautiful blend of Mughal and Rajput architecture (Entry Fee: INR 700 for Indians - consider only viewing the exterior if on a very tight budget).
*   **Evening (4:00 PM - 9:00 PM):**
    *   Explore Jantar Mantar, an astronomical observatory (Entry Fee: INR 100 for Indians).
    *   Walk to Bapu Bazaar for textiles and handicraf

# 📃 Travel Guide RAG Setup (ChromaDB)
```markdown
We now create a custom travel guide (e.g., for India), chunk it, and embed into a Chroma vector database.
```

In [6]:
india_guide = """
# India Travel Guide

## 🏨 Where to Stay (Budget Focus)
- Jaipur: Hostels near MI Road, Zostel Jaipur
- Delhi: Paharganj for budget, Karol Bagh for mid-range
- Rishikesh: Tapovan area for spiritual retreats

## 🚌 Transportation
- Use auto-rickshaws, metros, and local buses
- Intercity: Trains, sleeper buses, or budget flights
- Apps like Ola and Uber work in major cities

## 🍛 Street Food You Must Try
- Delhi: Chole Bhature, Golgappa, Parathas at Chandni Chowk
- Kolkata: Kathi rolls, Puchka
- Mumbai: Vada Pav, Pav Bhaji

## 🕌 Must-See Sights
- Agra: Taj Mahal
- Jaipur: Amer Fort, Hawa Mahal
- Varanasi: Ganga Aarti, Boat Rides
- Kerala: Backwaters, Houseboats

## ✅ Travel Tips
- Carry change, keep UPI apps handy
- Bargain at markets
- Dress modestly in temples

## ⚠️ Safety & Culture
- Avoid isolated areas at night
- Respect local customs and language
- Always remove shoes before entering homes or temples
"""

### 🧩 3.2 Chunk guide and preview


In [7]:
chunks = [chunk.strip() for chunk in india_guide.split("\n\n") if chunk.strip()]
print(f"✅ Total chunks created: {len(chunks)}\n")
for i, chunk in enumerate(chunks):
    print(f"[Chunk {i+1}]: {chunk}\n---")

✅ Total chunks created: 7

[Chunk 1]: # India Travel Guide
---
[Chunk 2]: ## 🏨 Where to Stay (Budget Focus)
- Jaipur: Hostels near MI Road, Zostel Jaipur
- Delhi: Paharganj for budget, Karol Bagh for mid-range
- Rishikesh: Tapovan area for spiritual retreats
---
[Chunk 3]: ## 🚌 Transportation
- Use auto-rickshaws, metros, and local buses
- Intercity: Trains, sleeper buses, or budget flights
- Apps like Ola and Uber work in major cities
---
[Chunk 4]: ## 🍛 Street Food You Must Try
- Delhi: Chole Bhature, Golgappa, Parathas at Chandni Chowk
- Kolkata: Kathi rolls, Puchka
- Mumbai: Vada Pav, Pav Bhaji
---
[Chunk 5]: ## 🕌 Must-See Sights
- Agra: Taj Mahal
- Jaipur: Amer Fort, Hawa Mahal
- Varanasi: Ganga Aarti, Boat Rides
- Kerala: Backwaters, Houseboats
---
[Chunk 6]: ## ✅ Travel Tips
- Carry change, keep UPI apps handy
- Bargain at markets
- Dress modestly in temples
---
[Chunk 7]: ## ⚠️ Safety & Culture
- Avoid isolated areas at night
- Respect local customs and language
- Always remove


## 4️⃣ ChromaDB + Gemini Embedding for Retrieval (RAG)

### 🔗 4.1 Setup embedding and ChromaDB collection

In [8]:
from google.api_core import retry
import chromadb
from chromadb import Documents, EmbeddingFunction

is_retriable = lambda e: isinstance(e, genai.errors.APIError) and e.code in {429, 503}

@retry.Retry(predicate=is_retriable, timeout=300.0)
def embed_chunk(text: str) -> list[float]:
    response = client.models.embed_content(
        model="models/text-embedding-004",
        contents=text,
        config=genai.types.EmbedContentConfig(task_type="retrieval_document"),
    )
    return response.embeddings[0].values

class GeminiEmbeddingFunction(EmbeddingFunction):
    def __call__(self, texts: Documents):
        return [embed_chunk(text) for text in texts]

chroma_client = chromadb.Client()
# Use get_or_create to avoid collection errors
collection = chroma_client.get_or_create_collection(
    name="india_guide",
    embedding_function=GeminiEmbeddingFunction()
)

collection.add(documents=chunks, ids=[f"guide_chunk_{i}" for i in range(len(chunks))])
print("✅ Guide added to ChromaDB.")

✅ Guide added to ChromaDB.


## 5️⃣ Ask RAG Questions Based on Indian Guide

### ❓ 5.1 Embed user query and retrieve context

In [9]:
@retry.Retry(predicate=is_retriable, timeout=300.0)
def embed_query(text: str) -> list[float]:
    response = client.models.embed_content(
        model="models/text-embedding-004",
        contents=text,
        config=genai.types.EmbedContentConfig(task_type="retrieval_query"),
    )
    return response.embeddings[0].values

user_question = "What are good local foods to try in Delhi?"
query_vector = embed_query(user_question)
results = collection.query(query_embeddings=[query_vector], n_results=3)
retrieved_chunks = results["documents"][0]
context = "\n".join(retrieved_chunks)

### 🧠 5.2 Ask Gemini to answer with retrieved info

In [10]:
rag_prompt = f"""
You are an India travel assistant AI.

Based on the following info, answer this:
---
{context}
---

Question: {user_question}
Answer:
"""
response = client.models.generate_content(
    model="gemini-2.0-flash",
    contents=rag_prompt
)
print("📌 RAG Answer:\n", response.text)

📌 RAG Answer:
 Based on the travel guide, good local foods to try in Delhi are:

*   Chole Bhature
*   Golgappa
*   Parathas at Chandni Chowk



# 🌐 RAG + Google Search Grounding
```markdown
We now enhance the RAG response with real-time Google Search grounding.
```

In [11]:
from google.genai.types import GenerateContentConfig, Tool, GoogleSearch

combined_prompt = f"""
You are an expert India travel assistant.

Use this internal guide + live web search to answer:

Guide Info:
{context}

Question: {user_question}
"""

config = GenerateContentConfig(
    temperature=0.5,
    max_output_tokens=1024,
    tools=[Tool(google_search=GoogleSearch())]
)

response = client.models.generate_content(
    model="gemini-2.0-flash",
    contents=combined_prompt,
    config=config
)
print("🌐 Combined Answer:\n", response.text)

🌐 Combined Answer:
 Based on the information from the India Travel Guide and the search results, here are some good local foods to try in Delhi:

*   **Chole Bhature:** A very famous dish, often enjoyed for breakfast or lunch.
*   **Golgappa (Pani Puri):** Crispy, hollow puris filled with spicy, tangy water, tamarind chutney, potatoes, and chickpeas.
*   **Parathas:** You can find some of the best-stuffed parathas in Delhi, especially in Paranthe Wali Gali in Old Delhi.
*   **Aloo Tikki:** Spiced mashed potatoes shaped into patties and fried until crispy.
*   **Kebabs:** Made from minced meat, marinated with spices, and grilled over an open flame.



## ✅ Summary
This notebook demonstrates India-focused Travel Agent AI using:
- 🧠 Chain-of-Thought & Few-shot for structured itinerary
- 🔍 ChromaDB + Gemini embeddings for RAG
- 🌐 Google Search Tool for live web knowledge

This approach can scale to any country, topic, or personal domain assistant. Feel free to extend the knowledge base or UI for a travel chatbot, planner app, or concierge platform.

---

*Built with 💙 for the Google Gen AI Capstone Project – 2025 Q1.*
